# End-to-End Single-Experiment Walkthrough

## 1. Title and purpose

This notebook demonstrates one full, operator-visible lifecycle for **one experiment** in the Thesis_ML framework:

1. confirm framework health,
2. inspect and select one shipped registry experiment,
3. dry-run it,
4. run it for real,
5. inspect outputs/artifacts/reports,
6. inspect artifact lineage in the SQLite registry,
7. summarize the run.

The walkthrough is intentionally explicit and command-driven so the operational process stays visible.
        

## 2. Environment and path setup

This section is intentionally explicit.

- Edit the path variables in the setup cell for your machine.
- Keep notebook outputs isolated under `outputs/notebook_demo/...`.
- Real experiment execution requires valid dataset + cache paths.
        

In [13]:
from __future__ import annotations

import json
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from IPython.display import JSON, Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "Thesis_ML").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected pyproject.toml and src/Thesis_ML). "
        "Set PROJECT_ROOT manually in the setup cell."
    )


def resolve_cli_command(cli_name: str, module_fallback: str) -> list[str]:
    resolved = shutil.which(cli_name)
    if resolved:
        return [resolved]
    return [sys.executable, "-m", module_fallback]


def format_command(command: list[str]) -> str:
    return " ".join(str(part) for part in command)


@dataclass
class CommandResult:
    command: list[str]
    returncode: int
    stdout: str
    stderr: str


def run_command(command: list[str], *, cwd: Path, check: bool = False) -> CommandResult:
    printable = format_command(command)
    print(f"$ {printable}")
    process = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd),
        capture_output=True,
        text=True,
    )
    print(f"[returncode] {process.returncode}")
    if process.stdout.strip():
        print("\n[stdout]\n" + process.stdout)
    if process.stderr.strip():
        print("\n[stderr]\n" + process.stderr)

    result = CommandResult(
        command=[str(part) for part in command],
        returncode=process.returncode,
        stdout=process.stdout,
        stderr=process.stderr,
    )
    if check and process.returncode != 0:
        raise RuntimeError(
            "Command failed.\n"
            f"Command: {printable}\n"
            f"Return code: {process.returncode}\n"
            f"Stdout:\n{process.stdout}\n"
            f"Stderr:\n{process.stderr}"
        )
    return result


def parse_campaign_field(stdout: str, field_name: str) -> str | None:
    prefix = f"- {field_name}:"
    for raw_line in stdout.splitlines():
        line = raw_line.strip()
        if line.startswith(prefix):
            return line.split(":", 1)[1].strip()
    return None


def latest_campaign_dir(output_root: Path) -> Path | None:
    campaigns_dir = output_root / "campaigns"
    if not campaigns_dir.exists():
        return None
    candidates = [p for p in campaigns_dir.iterdir() if p.is_dir()]
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]


def render_tree(root: Path, max_depth: int = 4, max_entries: int = 300) -> str:
    if not root.exists():
        return f"[missing] {root}"
    lines: list[str] = [str(root)]
    count = 0
    root = root.resolve()
    for path in sorted(root.rglob("*")):
        try:
            rel = path.relative_to(root)
        except Exception:
            continue
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        marker = "/" if path.is_dir() else ""
        lines.append(f"{indent}- {rel}{marker}")
        count += 1
        if count >= max_entries:
            lines.append("... [truncated]")
            break
    return "\n".join(lines)


def collect_key_files(search_roots: list[Path], key_names: set[str]) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            if path.name not in key_names:
                continue
            rows.append(
                {
                    "file_name": path.name,
                    "path": str(path.resolve()),
                    "size_bytes": int(path.stat().st_size),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["file_name", "path", "size_bytes"])
    return pd.DataFrame(rows).sort_values(["file_name", "path"]).reset_index(drop=True)


def safe_read_json(path: Path) -> dict | list:
    return json.loads(path.read_text(encoding="utf-8"))


def preview_csv(path: Path, n: int = 10) -> pd.DataFrame:
    return pd.read_csv(path).head(n)


pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

In [14]:
# REQUIRED USER CONFIGURATION (edit explicitly for your environment)
PROJECT_ROOT = find_project_root(Path.cwd())
REGISTRY_PATH = PROJECT_ROOT / "configs" / "decision_support_registry.json"

# Real-data paths (edit these explicitly before running sections 5 and 6)
DATASET_INDEX_CSV = PROJECT_ROOT / "Data" / "processed" / "dataset_index.csv"
DATA_ROOT = PROJECT_ROOT / "Data"

# Notebook-isolated paths
NOTEBOOK_RUN_ROOT = PROJECT_ROOT / "outputs" / "notebook_demo" / "end_to_end_experiment_walkthrough"
NOTEBOOK_CACHE_DIR = NOTEBOOK_RUN_ROOT / "cache"
NOTEBOOK_OUTPUT_ROOT = NOTEBOOK_RUN_ROOT / "campaign_outputs"
NOTEBOOK_REGISTRY_DIR = NOTEBOOK_RUN_ROOT / "registry"
NOTEBOOK_TMP_DIR = NOTEBOOK_RUN_ROOT / "tmp"

# Safety flag: set True only after you confirm DATASET_INDEX_CSV and DATA_ROOT are correct.
DATA_PATHS_CONFIRMED = True

# Optional: auto-build cache into NOTEBOOK_CACHE_DIR if missing (can be expensive).
AUTO_BUILD_NOTEBOOK_CACHE_IF_MISSING = False

for directory in [NOTEBOOK_RUN_ROOT, NOTEBOOK_CACHE_DIR, NOTEBOOK_OUTPUT_ROOT, NOTEBOOK_REGISTRY_DIR, NOTEBOOK_TMP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

config_rows = [
    ("PROJECT_ROOT", PROJECT_ROOT),
    ("REGISTRY_PATH", REGISTRY_PATH),
    ("DATASET_INDEX_CSV", DATASET_INDEX_CSV),
    ("DATA_ROOT", DATA_ROOT),
    ("NOTEBOOK_RUN_ROOT", NOTEBOOK_RUN_ROOT),
    ("NOTEBOOK_CACHE_DIR", NOTEBOOK_CACHE_DIR),
    ("NOTEBOOK_OUTPUT_ROOT", NOTEBOOK_OUTPUT_ROOT),
    ("NOTEBOOK_REGISTRY_DIR", NOTEBOOK_REGISTRY_DIR),
    ("NOTEBOOK_TMP_DIR", NOTEBOOK_TMP_DIR),
]

setup_df = pd.DataFrame(config_rows, columns=["setting", "value"])
display(setup_df)

if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry path does not exist: {REGISTRY_PATH}")

print(f"DATA_PATHS_CONFIRMED = {DATA_PATHS_CONFIRMED} (set to True before execution sections)")

,setting,value
0,PROJECT_ROOT,d:\Khaled\Projects\VScode\Thesis
1,REGISTRY_PATH,d:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json
2,DATASET_INDEX_CSV,d:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv
3,DATA_ROOT,d:\Khaled\Projects\VScode\Thesis\Data
4,NOTEBOOK_RUN_ROOT,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough
5,NOTEBOOK_CACHE_DIR,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache
6,NOTEBOOK_OUTPUT_ROOT,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs
7,NOTEBOOK_REGISTRY_DIR,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\registry
8,NOTEBOOK_TMP_DIR,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\tmp


DATA_PATHS_CONFIRMED = True (set to True before execution sections)


## 3. Framework health check

This runs the project acceptance smoke path (`scripts/acceptance_smoke.py`) and fails early if the framework is not healthy.
        

In [15]:
ACCEPTANCE_SMOKE_SCRIPT = PROJECT_ROOT / "scripts" / "acceptance_smoke.py"
if not ACCEPTANCE_SMOKE_SCRIPT.exists():
    raise FileNotFoundError(f"Acceptance smoke script not found: {ACCEPTANCE_SMOKE_SCRIPT}")

health_result = run_command(
    [sys.executable, str(ACCEPTANCE_SMOKE_SCRIPT)],
    cwd=PROJECT_ROOT,
    check=False,
)
HEALTH_CHECK_OK = health_result.returncode == 0
print(f"HEALTH_CHECK_OK={HEALTH_CHECK_OK}")

if not HEALTH_CHECK_OK:
    raise RuntimeError(
        "Framework health check failed. Fix acceptance smoke failures before continuing this walkthrough."
    )

$ d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\python.exe d:\Khaled\Projects\VScode\Thesis\scripts\acceptance_smoke.py
[returncode] 0

[stdout]
Template compile check: non-runnable template (no enabled rows).
No additional workbook sample assets detected in templates/.
$ thesisml-workbook --output C:\Users\Khaled\AppData\Local\Temp\thesisml-acceptance-_dhxfs6b\generated_workbook.xlsx
Created workbook: C:\Users\Khaled\AppData\Local\Temp\thesisml-acceptance-_dhxfs6b\generated_workbook.xlsx
Sheet order valid: True
Missing required sheets: None
Legacy required sheets present: True
New sheets present: True
Sheet count: 22
Data validations found: 56
Experiment_Definitions columns valid: True
Run_Log new columns present: True
Required named lists present: True
Missing named lists: None
Experiment_Ready formula present: True
Confirmatory formulas present: True
Dashboard formulas present: True
Stage vocabulary consistent: True
Stage rows detected: 7
$ thesisml-run-decision-support --registry 

## 4. Inspect the shipped registry

This section loads the committed registry, shows available experiments, selects one experiment ID, and writes a notebook-local single-experiment registry file.
        

In [16]:
registry_payload = safe_read_json(REGISTRY_PATH)
experiments = list(registry_payload.get("experiments", []))
if not experiments:
    raise RuntimeError(f"No experiments found in registry: {REGISTRY_PATH}")

registry_table = pd.DataFrame(
    [
        {
            "experiment_id": exp.get("experiment_id"),
            "title": exp.get("title"),
            "stage": exp.get("stage"),
            "decision_id": exp.get("decision_id"),
            "executable_now": exp.get("executable_now"),
            "execution_status": exp.get("execution_status"),
            "blocked_reasons": "; ".join(exp.get("blocked_reasons", []) or []),
        }
        for exp in experiments
    ]
).sort_values("experiment_id")

display(registry_table)
print(f"Registry schema_version: {registry_payload.get('schema_version')}")

,experiment_id,title,stage,decision_id,executable_now,execution_status,blocked_reasons
0,E01,Target granularity experiment,Stage 1 - Target lock,D01,True,partially_blocked,binary valence-like target not implemented in thesisml-run-experiment target choices.
1,E02,Task pooling experiment,Stage 1 - Target lock,D01,True,executable,
2,E03,Modality pooling experiment,Stage 1 - Target lock,D01,True,executable,
3,E04,Split-strength stress test,Stage 2 - Split lock,D02,True,partially_blocked,record-wise random split mode is not implemented in runner.
4,E05,Cross-person transfer design,Stage 2 - Split lock,D02,True,executable,
5,E06,Linear model family comparison,Stage 3 - Model lock,D03,True,executable,
6,E07,Class weighting experiment,Stage 3 - Model lock,D04,False,blocked,No class weighting flag/parameter exposed in thesisml-run-experiment.
7,E08,Hyperparameter strategy experiment,Stage 3 - Model lock,D05,False,blocked,No nested tuning strategy option is implemented in current runner.
8,E09,Whole-brain versus ROI experiment,Stage 4 - Feature/preprocessing lock,D06,False,blocked,Feature-space selection (whole-brain vs ROI) is not parameterized in current experiment path.
9,E10,Dimensionality reduction experiment,Stage 4 - Feature/preprocessing lock,D06,False,blocked,No dimensionality reduction strategy option is exposed in runner pipeline.


Registry schema_version: 1.0


In [17]:
# Choose exactly one experiment for this walkthrough.
SELECTED_EXPERIMENT_ID = "E02"  # <-- edit as needed

experiment_map = {str(exp.get("experiment_id")): exp for exp in experiments}
if SELECTED_EXPERIMENT_ID not in experiment_map:
    raise ValueError(
        f"Experiment '{SELECTED_EXPERIMENT_ID}' is not present in {REGISTRY_PATH}. "
        f"Available: {sorted(experiment_map)}"
    )

selected_experiment = experiment_map[SELECTED_EXPERIMENT_ID]
display(pd.DataFrame([selected_experiment]).T.rename(columns={0: "value"}))

single_registry_payload = dict(registry_payload)
single_registry_payload["experiments"] = [selected_experiment]
SINGLE_REGISTRY_PATH = NOTEBOOK_REGISTRY_DIR / f"single_experiment_{SELECTED_EXPERIMENT_ID}.json"
SINGLE_REGISTRY_PATH.write_text(json.dumps(single_registry_payload, indent=2) + "\n", encoding="utf-8")

print(f"Wrote single-experiment registry: {SINGLE_REGISTRY_PATH}")

,value
experiment_id,E02
title,Task pooling experiment
stage,Stage 1 - Target lock
decision_id,D01
manipulated_factor,Pooling strategy by task
fixed_controls,"coarse_affect target, ridge model, within_subject_loso_session split, seed=42."
dataset_scope,Internal BAS2 repeated-session index.
target_definition,coarse_affect
split_logic,"within_subject_loso_session, per subject."
models,[ridge]


Wrote single-experiment registry: d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\registry\single_experiment_E02.json


## 5. Dry-run the selected experiment

This uses the canonical decision-support CLI path with explicit `--registry`, `--index-csv`, `--data-root`, `--cache-dir`, and `--output-root`.
        

In [18]:
if not DATA_PATHS_CONFIRMED:
    raise RuntimeError(
        "Set DATA_PATHS_CONFIRMED=True in the setup cell after explicitly validating DATASET_INDEX_CSV and DATA_ROOT."
    )
if not DATASET_INDEX_CSV.exists():
    raise FileNotFoundError(f"Dataset index CSV not found: {DATASET_INDEX_CSV}")
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data root not found: {DATA_ROOT}")

DECISION_SUPPORT_CMD = resolve_cli_command(
    "thesisml-run-decision-support", "Thesis_ML.cli.decision_support"
)
print(f"Decision-support command base: {DECISION_SUPPORT_CMD}")

dry_run_command = DECISION_SUPPORT_CMD + [
    "--registry",
    str(SINGLE_REGISTRY_PATH),
    "--experiment-id",
    SELECTED_EXPERIMENT_ID,
    "--index-csv",
    str(DATASET_INDEX_CSV),
    "--data-root",
    str(DATA_ROOT),
    "--cache-dir",
    str(NOTEBOOK_CACHE_DIR),
    "--output-root",
    str(NOTEBOOK_OUTPUT_ROOT),
    "--dry-run",
]

DRY_RUN_RESULT = run_command(dry_run_command, cwd=PROJECT_ROOT, check=False)
DRY_RUN_SUCCEEDED = DRY_RUN_RESULT.returncode == 0
print(f"DRY_RUN_SUCCEEDED={DRY_RUN_SUCCEEDED}")
if not DRY_RUN_SUCCEEDED:
    raise RuntimeError("Dry-run failed. Resolve the error before continuing.")

dry_run_campaign_root_raw = parse_campaign_field(DRY_RUN_RESULT.stdout, "campaign_root")
DRY_RUN_CAMPAIGN_ROOT = Path(dry_run_campaign_root_raw) if dry_run_campaign_root_raw else latest_campaign_dir(NOTEBOOK_OUTPUT_ROOT)
print(f"DRY_RUN_CAMPAIGN_ROOT={DRY_RUN_CAMPAIGN_ROOT}")

Decision-support command base: ['d:\\Khaled\\Projects\\VScode\\Thesis\\.venv\\Scripts\\thesisml-run-decision-support.EXE']
$ d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE --registry d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\registry\single_experiment_E02.json --experiment-id E02 --index-csv d:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root d:\Khaled\Projects\VScode\Thesis\Data --cache-dir d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache --output-root d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs --dry-run
[returncode] 0

[stdout]
Campaign complete
- campaign_id: 20260314_041859_929889
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\20260314_041859_929889
- selected_experiments: E02
- status

## 6. Run the experiment for real

This executes the same selected experiment without `--dry-run`.

If notebook-local cache is missing, this section can optionally build it first using `thesisml-cache-features`.
        

In [19]:
cache_manifest_path = NOTEBOOK_CACHE_DIR / "cache_manifest.csv"
if not cache_manifest_path.exists():
    if AUTO_BUILD_NOTEBOOK_CACHE_IF_MISSING:
        CACHE_FEATURES_CMD = resolve_cli_command("thesisml-cache-features", "Thesis_ML.features.nifti_features")
        cache_build_command = CACHE_FEATURES_CMD + [
            "--index-csv",
            str(DATASET_INDEX_CSV),
            "--data-root",
            str(DATA_ROOT),
            "--cache-dir",
            str(NOTEBOOK_CACHE_DIR),
        ]
        print("Cache manifest not found; building notebook-local cache...")
        cache_result = run_command(cache_build_command, cwd=PROJECT_ROOT, check=False)
        if cache_result.returncode != 0:
            raise RuntimeError("Failed to build notebook-local cache. See command output above.")
    else:
        print(
            "Notebook cache manifest is missing; proceeding with full run. "
            "The experiment pipeline will build or reuse feature cache as needed.\n"
            f"Expected manifest path: {cache_manifest_path}"
        )

full_run_command = DECISION_SUPPORT_CMD + [
    "--registry",
    str(SINGLE_REGISTRY_PATH),
    "--experiment-id",
    SELECTED_EXPERIMENT_ID,
    "--index-csv",
    str(DATASET_INDEX_CSV),
    "--data-root",
    str(DATA_ROOT),
    "--cache-dir",
    str(NOTEBOOK_CACHE_DIR),
    "--output-root",
    str(NOTEBOOK_OUTPUT_ROOT),
]

FULL_RUN_RESULT = run_command(full_run_command, cwd=PROJECT_ROOT, check=False)
FULL_RUN_SUCCEEDED = FULL_RUN_RESULT.returncode == 0
print(f"FULL_RUN_SUCCEEDED={FULL_RUN_SUCCEEDED}")
if not FULL_RUN_SUCCEEDED:
    raise RuntimeError("Full run failed. See command output above.")

full_campaign_root_raw = parse_campaign_field(FULL_RUN_RESULT.stdout, "campaign_root")
FULL_RUN_CAMPAIGN_ROOT = Path(full_campaign_root_raw) if full_campaign_root_raw else latest_campaign_dir(NOTEBOOK_OUTPUT_ROOT)
if FULL_RUN_CAMPAIGN_ROOT is None:
    raise RuntimeError("Could not locate full-run campaign root.")

full_manifest_raw = parse_campaign_field(FULL_RUN_RESULT.stdout, "manifest")
FULL_RUN_CAMPAIGN_MANIFEST_PATH = (
    Path(full_manifest_raw) if full_manifest_raw else FULL_RUN_CAMPAIGN_ROOT / "campaign_manifest.json"
)
print(f"FULL_RUN_CAMPAIGN_ROOT={FULL_RUN_CAMPAIGN_ROOT}")
print(f"FULL_RUN_CAMPAIGN_MANIFEST_PATH={FULL_RUN_CAMPAIGN_MANIFEST_PATH}")

Notebook cache manifest is missing; proceeding with full run. The experiment pipeline will build or reuse feature cache as needed.
Expected manifest path: d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache\cache_manifest.csv
$ d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE --registry d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\registry\single_experiment_E02.json --experiment-id E02 --index-csv d:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root d:\Khaled\Projects\VScode\Thesis\Data --cache-dir d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache --output-root d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs
[returncode] 0

[stdout]
Campaign complete
- campaign_id: 20260314_041902_744546
- campaign_root: D:\Khaled\Projects\VScode\Thesis\output

## 7. Inspect outputs on disk

This shows output structure and highlights key files (status/config/metrics/predictions/spatial reports/aggregation/report exports).
        

In [20]:
if not FULL_RUN_CAMPAIGN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Campaign manifest not found: {FULL_RUN_CAMPAIGN_MANIFEST_PATH}")

CAMPAIGN_MANIFEST = safe_read_json(FULL_RUN_CAMPAIGN_MANIFEST_PATH)
display(JSON(CAMPAIGN_MANIFEST, expanded=False))

print("Campaign output tree:\n")
print(render_tree(FULL_RUN_CAMPAIGN_ROOT, max_depth=5, max_entries=500))

search_roots = [FULL_RUN_CAMPAIGN_ROOT]
for root_str in CAMPAIGN_MANIFEST.get("experiment_roots", {}).values():
    search_roots.append(Path(root_str))

KEY_FILENAMES = {
    "campaign_manifest.json",
    "decision_support_summary.csv",
    "decision_recommendations.md",
    "result_aggregation.json",
    "summary_outputs_export.csv",
    "run_log_export.csv",
    "run_status.json",
    "config.json",
    "metrics.json",
    "fold_metrics.csv",
    "predictions.csv",
    "spatial_compatibility_report.json",
    "interpretability_summary.json",
    "interpretability_fold_explanations.csv",
}

KEY_FILES_DF = collect_key_files(search_roots, KEY_FILENAMES)
if KEY_FILES_DF.empty:
    print("No key output files were found under the expected roots.")
else:
    display(KEY_FILES_DF)

<IPython.core.display.JSON object>

Campaign output tree:

D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\20260314_041902_744546
- campaign_manifest.json
- decision_recommendations.md
- decision_support_summary.csv
- result_aggregation.json
- run_log_export.csv
- stage1_target_lock_summary.csv
- stage1_target_lock_summary.md
- stage1_target_lock_summary_decision_notes.md
- stage2_split_lock_summary.csv
- stage2_split_lock_summary.md
- stage3_model_lock_summary.csv
- stage3_model_lock_summary.md
- stage4_feature_lock_summary.csv
- stage4_feature_lock_summary.md
- stage5_confirmatory_summary.csv
- stage5_confirmatory_summary.md
- stage6_robustness_summary.csv
- stage6_robustness_summary.md
- stage7_exploratory_summary.csv
- stage7_exploratory_summary.md
- summary_outputs_export.csv


,file_name,path,size_bytes
0,campaign_manifest.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\...,5480
1,config.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,3011
2,config.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,3011
3,config.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,3017
4,config.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,3021
...,...,...,...
65,spatial_compatibility_report.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,21838
66,spatial_compatibility_report.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,26254
67,spatial_compatibility_report.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,26254
68,spatial_compatibility_report.json,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,26254


## 8. Inspect JSON outputs

At minimum: `run_status.json`, `metrics.json`, and `spatial_compatibility_report.json`.
        

In [21]:
def first_file_path(file_name: str) -> Path | None:
    subset = KEY_FILES_DF[KEY_FILES_DF["file_name"] == file_name]
    if subset.empty:
        return None
    return Path(subset.iloc[0]["path"])

run_status_path = first_file_path("run_status.json")
metrics_path = first_file_path("metrics.json")
spatial_report_path = first_file_path("spatial_compatibility_report.json")

print(f"run_status_path={run_status_path}")
print(f"metrics_path={metrics_path}")
print(f"spatial_report_path={spatial_report_path}")

if run_status_path and run_status_path.exists():
    display(Markdown("### run_status.json"))
    display(JSON(safe_read_json(run_status_path), expanded=False))
else:
    print("run_status.json not found.")

if metrics_path and metrics_path.exists():
    display(Markdown("### metrics.json"))
    display(JSON(safe_read_json(metrics_path), expanded=False))
else:
    print("metrics.json not found.")

if spatial_report_path and spatial_report_path.exists():
    display(Markdown("### spatial_compatibility_report.json"))
    display(JSON(safe_read_json(spatial_report_path), expanded=False))
else:
    print("spatial_compatibility_report.json not found.")

run_status_path=D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\20260314_041902_744546\reports\ds_E02_pooled_tasks_001_20260314_041902_744546\run_status.json
metrics_path=D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\20260314_041902_744546\reports\ds_E02_pooled_tasks_001_20260314_041902_744546\metrics.json
spatial_report_path=D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\20260314_041902_744546\reports\ds_E02_pooled_tasks_001_20260314_041902_744546\spatial_compatibility_report.json


### run_status.json

<IPython.core.display.JSON object>

### metrics.json

<IPython.core.display.JSON object>

### spatial_compatibility_report.json

<IPython.core.display.JSON object>

## 9. Inspect tabular outputs

This previews key CSV outputs (predictions, fold metrics, and campaign-level summary tables), with graceful handling when files are missing.
        

In [22]:
TABULAR_FILE_ORDER = [
    "predictions.csv",
    "fold_metrics.csv",
    "decision_support_summary.csv",
    "run_log_export.csv",
    "summary_outputs_export.csv",
]

for file_name in TABULAR_FILE_ORDER:
    file_path = first_file_path(file_name)
    display(Markdown(f"### {file_name}"))
    if file_path is None or not file_path.exists():
        print("Not found.")
        continue
    try:
        preview = preview_csv(file_path, n=12)
    except Exception as exc:
        print(f"Failed to read {file_path}: {exc}")
        continue
    print(f"Path: {file_path}")
    print(f"Rows previewed: {len(preview)}")
    display(preview)

### predictions.csv

Path: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\20260314_041902_744546\reports\ds_E02_pooled_tasks_001_20260314_041902_744546\predictions.csv
Rows previewed: 12


,fold,sample_id,y_true,y_pred,decision_value,decision_vector,proba_value,proba_vector,subject,session,bas,task,modality,emotion,coarse_affect,experiment_mode,train_subject,test_subject
0,0,sub-001_ses-01_BAS2_run-1_beta-1,negative,negative,NaN,"[0.447099506855011, -1.8623688220977783, 0.4152665138244629]",NaN,NaN,sub-001,ses-01,BAS2,passive,audio,anger,negative,within_subject_loso_session,NaN,NaN
1,0,sub-001_ses-01_BAS2_run-1_beta-2,negative,negative,NaN,"[0.20746883749961853, -1.2015929222106934, -0.005871795117855072]",NaN,NaN,sub-001,ses-01,BAS2,passive,video,anger,negative,within_subject_loso_session,NaN,NaN
2,0,sub-001_ses-01_BAS2_run-1_beta-3,negative,negative,NaN,"[1.2222316265106201, -1.3486223220825195, -0.8736132979393005]",NaN,NaN,sub-001,ses-01,BAS2,passive,audiovisual,anger,negative,within_subject_loso_session,NaN,NaN
3,0,sub-001_ses-01_BAS2_run-1_beta-4,negative,positive,NaN,"[-0.9825498461723328, -0.7638826370239258, 0.7464228272438049]",NaN,NaN,sub-001,ses-01,BAS2,passive,audio,anxiety,negative,within_subject_loso_session,NaN,NaN
4,0,sub-001_ses-01_BAS2_run-1_beta-5,negative,negative,NaN,"[0.00376022607088089, -0.16825222969055176, -0.8355106711387634]",NaN,NaN,sub-001,ses-01,BAS2,passive,video,anxiety,negative,within_subject_loso_session,NaN,NaN
5,0,sub-001_ses-01_BAS2_run-1_beta-6,negative,negative,NaN,"[0.2554177939891815, -1.2502042055130005, -0.005213834345340729]",NaN,NaN,sub-001,ses-01,BAS2,passive,audiovisual,anxiety,negative,within_subject_loso_session,NaN,NaN
6,0,sub-001_ses-01_BAS2_run-1_beta-7,negative,negative,NaN,"[0.1331639289855957, -0.8419659733772278, -0.29119181632995605]",NaN,NaN,sub-001,ses-01,BAS2,passive,audio,disgust,negative,within_subject_loso_session,NaN,NaN
7,0,sub-001_ses-01_BAS2_run-1_beta-8,negative,negative,NaN,"[0.7495250105857849, -1.7154967784881592, -0.034038037061691284]",NaN,NaN,sub-001,ses-01,BAS2,passive,video,disgust,negative,within_subject_loso_session,NaN,NaN
8,0,sub-001_ses-01_BAS2_run-1_beta-9,negative,negative,NaN,"[0.8203448057174683, -1.4961252212524414, -0.32422012090682983]",NaN,NaN,sub-001,ses-01,BAS2,passive,audiovisual,disgust,negative,within_subject_loso_session,NaN,NaN
9,0,sub-001_ses-01_BAS2_run-1_beta-10,positive,negative,NaN,"[2.4461636543273926, -2.0683813095092773, -1.377781867980957]",NaN,NaN,sub-001,ses-01,BAS2,passive,audio,happiness,positive,within_subject_loso_session,NaN,NaN


### fold_metrics.csv

Path: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\20260314_041902_744546\reports\ds_E02_pooled_tasks_001_20260314_041902_744546\fold_metrics.csv
Rows previewed: 12


...

Path: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\20260314_041902_744546\run_log_export.csv
Rows previewed: 8


,Run_ID,Experiment_ID,Run_Date,Dataset_Name,Data_Subset,Code_Commit_or_Version,Config_File_or_Path,Random_Seed,Target,Split_ID_or_Fold_Definition,...,Primary_Metric_Value,Secondary_Metric_1,Secondary_Metric_2,Robustness_Output_Summary,Result_Summary,Preliminary_Interpretation,Reviewed,Used_in_Thesis,Artifact_Path,Notes
0,ds_E02_pooled_tasks_001_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-001;task=all_tasks;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.670565,0.666982,0.712801,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
1,ds_E02_pooled_tasks_002_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-002;task=all_tasks;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.573269,0.574728,0.635534,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
2,ds_E02_task_specific_001_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-001;task=emo;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.833333,0.825018,0.883041,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
3,ds_E02_task_specific_002_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-001;task=passive;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.383041,0.377799,0.475634,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
4,ds_E02_task_specific_003_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-001;task=recog;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.824561,0.823118,0.865497,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
5,ds_E02_task_specific_004_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-002;task=emo;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.679952,0.686209,0.761675,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
6,ds_E02_task_specific_005_20260314_041902_744546,E02,2026-03-14,Internal BAS2,subject=sub-002;task=passive;modality=all_modalities,327271fb66986e9146771ba04916708147067559,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,42,coarse_affect,within_subject_loso_session,...,0.349034,0.333382,0.455717,NaN,completed,NaN,No,No,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,NaN
7,ds_E02_task_specific_006_20260314_041902_744546,E02,2026-03-14,Int

### summary_outputs_export.csv

Path: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\20260314_041902_744546\summary_outputs_export.csv
Rows previewed: 9


,summary_type,summary_key,primary_metric_name,primary_metric_value,run_id,experiment_id,start_section,end_section,model,cv,target,xai_method,report_path,notes
0,best_full_pipeline,rank_1,balanced_accuracy,0.833333,ds_E02_task_specific_001_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,xai_status=performed
1,best_full_pipeline,rank_2,balanced_accuracy,0.824561,ds_E02_task_specific_003_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,xai_status=performed
2,best_full_pipeline,rank_3,balanced_accuracy,0.747585,ds_E02_task_specific_006_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,xai_status=performed
3,best_full_pipeline,rank_4,balanced_accuracy,0.679952,ds_E02_task_specific_004_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,xai_status=performed
4,best_full_pipeline,rank_5,balanced_accuracy,0.670565,ds_E02_pooled_tasks_001_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,xai_status=performed
5,section_effect,dataset_selection->evaluation,mean_primary_metric_value,0.632667,ds_E02_task_specific_001_20260314_041902_744546,E02,dataset_selection,evaluation,ridge,within_subject_loso_session,coarse_affect,linear_coefficients_stability,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,n_runs=8; best_primary_metric=0.8333333333333334
6,pattern_by_model,ridge,mean_primary_metric_value,0.632667,ds_E02_task_specific_001_20260314_041902_744546,E02,NaN,NaN,ridge,NaN,NaN,linear_coefficients_stability,NaN,n_runs=8; best_primary_metric=0.8333333333333334
7,pattern_by_cv,within_subject_loso_session,mean_primary_metric_value,0.632667,ds_E02_task_specific_001_20260314_041902_744546,E02,NaN,NaN,NaN,within_subject_loso_session,NaN,NaN,NaN,n_runs=8; best_primary_metric=0.8333333333333334
8,xai_method_effect,linear_coefficients_stability,mean_primary_metric_value,0.632667,ds_E02_task_specific_001_20260314_041902_744546,E02,NaN,NaN,NaN,NaN,NaN,linear_coefficients_stability,NaN,n_runs=8; performed=8; not_performed=0; unknown=0


## 10. Inspect artifact lineage

This reads the artifact registry SQLite DB, lists tables, previews artifact records, and displays upstream lineage links.
        

In [23]:
artifact_registry_path = Path(
    CAMPAIGN_MANIFEST.get("artifact_registry_path", NOTEBOOK_OUTPUT_ROOT / "artifact_registry.sqlite3")
)
print(f"artifact_registry_path={artifact_registry_path}")

if not artifact_registry_path.exists():
    print("Artifact registry database not found.")
else:
    with sqlite3.connect(str(artifact_registry_path)) as connection:
        tables_df = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection
        )
        display(Markdown("### Registry tables"))
        display(tables_df)

        artifacts_df = pd.read_sql_query(
            """
            SELECT artifact_id, artifact_type, run_id, status, path, upstream_artifact_ids, created_at
            FROM artifacts
            ORDER BY created_at DESC, artifact_id DESC
            LIMIT 50
            """,
            connection,
        )
        display(Markdown("### Artifact records (latest 50)"))
        display(artifacts_df)

        type_counts_df = pd.read_sql_query(
            """
            SELECT artifact_type, COUNT(*) AS n_records
            FROM artifacts
            GROUP BY artifact_type
            ORDER BY n_records DESC, artifact_type ASC
            """,
            connection,
        )
        display(Markdown("### Artifact type counts"))
        display(type_counts_df)

    lineage_edges: list[dict[str, str]] = []
    for _, row in artifacts_df.iterrows():
        raw_upstream = row.get("upstream_artifact_ids")
        try:
            upstream_ids = json.loads(raw_upstream) if raw_upstream else []
        except Exception:
            upstream_ids = []
        for upstream_id in upstream_ids:
            lineage_edges.append(
                {
                    "upstream_artifact_id": str(upstream_id),
                    "downstream_artifact_id": str(row["artifact_id"]),
                    "downstream_type": str(row["artifact_type"]),
                    "downstream_run_id": str(row["run_id"]),
                }
            )

    display(Markdown("### Lineage edges (upstream -> downstream)"))
    if lineage_edges:
        display(pd.DataFrame(lineage_edges))
    else:
        print("No upstream lineage edges recorded in previewed rows.")

artifact_registry_path=D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\artifact_registry.sqlite3


### Registry tables

,name
0,artifacts


### Artifact records (latest 50)

,artifact_id,artifact_type,run_id,status,path,upstream_artifact_ids,created_at
0,metrics_bundle_7b1efa7ec207e0f3e7cd6ddc,metrics_bundle,20260314_041902_744546,created,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\...,[],2026-03-14T03:48:43+00:00
1,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_1d81ed9002173a8f46810091"",""feature_matrix_bundle_c24cbb5430ea8bc55f84c73f"",""interpretability_bundle_...",2026-03-14T03:48:43+00:00
2,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_ad3581ff3e9892ed1bd59bdb"",""feature_matrix_bundle_1140335633834997503fff63"",""interpretability_bundle_...",2026-03-14T03:46:31+00:00
3,experiment_report_fb1522eb9b2926f8599dab25,experiment_report,ds_E02_task_specific_004_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_5550df9dbfd6129eb9f0c589"",""feature_matrix_bundle_a8066dc591102a5b0eb4c729"",""interpretability_bundle_...",2026-03-14T03:44:25+00:00
4,experiment_report_6edf358fe733333cac993d7c,experiment_report,ds_E02_task_specific_003_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_1bf4859f7b9d1a9eced2ea4d"",""feature_matrix_bundle_85b1f1f39b9d7a1abed076b1"",""interpretability_bundle_...",2026-03-14T03:42:12+00:00
5,experiment_report_fbd74117839160f0441ffa3c,experiment_report,ds_E02_task_specific_002_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_211281a4e45c269c8400c6dd"",""feature_matrix_bundle_3e3f9fb9cb68d68494bbafde"",""interpretability_bundle_...",2026-03-14T03:40:41+00:00
6,experiment_report_2e7c5159a9780da748c94791,experiment_report,ds_E02_task_specific_001_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_0b96be87aa5dca7733eb524a"",""feature_matrix_bundle_ae4eba19fae7cb0d624ffef1"",""interpretability_bundle_...",2026-03-14T03:39:05+00:00
7,experiment_report_290a123a674b1ffa3ee7719e,experiment_report,ds_E02_pooled_tasks_002_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_46d65141c4df7df14cc1b486"",""feature_matrix_bundle_01116896e206bb4b0be3a512"",""interpretability_bundle_...",2026-03-14T03:37:17+00:00
8,experiment_report_c42ee70b6c9ac2ffeb06aaff,experiment_report,ds_E02_pooled_tasks_001_20260314_041902_744546,completed,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\E02\202603...,"[""feature_cache_1b236bdc1c8e02d9caf57236"",""feature_matrix_bundle_6ec55c8bbbb0733d305f0dc8"",""interpretability_bundle_...",2026-03-14T03:31:29+00:00
9,metrics_bundle_35a04e05ff897fea34cad76f,metrics_bundle,20260314_041859_929889,created,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\...,[],2026-03-14T03:19:00+00:00


### Artifact type counts

,artifact_type,n_records
0,experiment_report,24
1,metrics_bundle,3


### Lineage edges (upstream -> downstream)

,upstream_artifact_id,downstream_artifact_id,downstream_type,downstream_run_id
0,feature_cache_1d81ed9002173a8f46810091,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546
1,feature_matrix_bundle_c24cbb5430ea8bc55f84c73f,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546
2,interpretability_bundle_0a657a1f06cd3618f6de47d5,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546
3,metrics_bundle_f347d05c3f6ffa14ef56408c,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546
4,experiment_report_fc14fd3ef8a5afa8ebc2e31a,experiment_report_2e41b8bd62e7dee5c7e272c7,experiment_report,ds_E02_task_specific_006_20260314_041902_744546
5,feature_cache_ad3581ff3e9892ed1bd59bdb,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546
6,feature_matrix_bundle_1140335633834997503fff63,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546
7,interpretability_bundle_9bace7a83a4f7c9c7c7e9fd1,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546
8,metrics_bundle_21e405a71c70739953f8eafa,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546
9,experiment_report_4d20121897b508ad38355d5b,experiment_report_17013cd718ac9c39e1ed47ac,experiment_report,ds_E02_task_specific_005_20260314_041902_744546


## 11. Notebook summary

This builds a concise end-state summary of the walkthrough run.
        

In [24]:
summary_payload = {
    "selected_experiment_id": SELECTED_EXPERIMENT_ID,
    "project_root": str(PROJECT_ROOT),
    "single_registry_path": str(SINGLE_REGISTRY_PATH),
    "dataset_index_csv": str(DATASET_INDEX_CSV),
    "data_root": str(DATA_ROOT),
    "cache_dir": str(NOTEBOOK_CACHE_DIR),
    "output_root": str(NOTEBOOK_OUTPUT_ROOT),
    "dry_run_succeeded": bool(globals().get("DRY_RUN_SUCCEEDED", False)),
    "full_run_succeeded": bool(globals().get("FULL_RUN_SUCCEEDED", False)),
    "dry_run_campaign_root": str(globals().get("DRY_RUN_CAMPAIGN_ROOT", "")),
    "full_run_campaign_root": str(globals().get("FULL_RUN_CAMPAIGN_ROOT", "")),
    "campaign_manifest_path": str(globals().get("FULL_RUN_CAMPAIGN_MANIFEST_PATH", "")),
    "artifact_registry_path": str(globals().get("artifact_registry_path", "")),
    "key_file_counts": (
        KEY_FILES_DF["file_name"].value_counts().sort_index().to_dict()
        if "KEY_FILES_DF" in globals() and not KEY_FILES_DF.empty
        else {}
    ),
    "key_metric_paths": {
        "metrics_json": str(metrics_path) if "metrics_path" in globals() and metrics_path else None,
        "fold_metrics_csv": str(first_file_path("fold_metrics.csv")) if "first_file_path" in globals() else None,
        "predictions_csv": str(first_file_path("predictions.csv")) if "first_file_path" in globals() else None,
        "spatial_compatibility_report_json": str(spatial_report_path) if "spatial_report_path" in globals() and spatial_report_path else None,
    },
}

display(pd.DataFrame(summary_payload.items(), columns=["field", "value"]))
display(Markdown("### Summary JSON"))
display(JSON(summary_payload, expanded=False))

,field,value
0,selected_experiment_id,E02
1,project_root,d:\Khaled\Projects\VScode\Thesis
2,single_registry_path,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\registry\single_experiment_...
3,dataset_index_csv,d:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv
4,data_root,d:\Khaled\Projects\VScode\Thesis\Data
5,cache_dir,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache
6,output_root,d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs
7,dry_run_succeeded,True
8,full_run_succeeded,True
9,dry_run_campaign_root,D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs\campaigns\...


### Summary JSON

<IPython.core.display.JSON object>

## 12. Optional workbook-driven appendix

**Optional section:** this demonstrates workbook generation/validation and shows why the shipped workbook template is non-runnable by default.

It does **not** assume workbook execution is possible without enabling/populating executable rows.
        

In [25]:
from Thesis_ML.orchestration.workbook_compiler import compile_workbook_file
from Thesis_ML.workbook.validation import validate_workbook

WORKBOOK_CMD = resolve_cli_command("thesisml-workbook", "Thesis_ML.cli.workbook")
optional_workbook_dir = NOTEBOOK_TMP_DIR / "workbook_appendix"
optional_workbook_dir.mkdir(parents=True, exist_ok=True)
optional_workbook_path = optional_workbook_dir / "generated_template.xlsx"

workbook_result = run_command(
    WORKBOOK_CMD + ["--output", str(optional_workbook_path)],
    cwd=PROJECT_ROOT,
    check=False,
)
if workbook_result.returncode != 0:
    raise RuntimeError("Workbook generation command failed.")

workbook_summary = validate_workbook(optional_workbook_path)
display(Markdown("### Generated workbook validation summary"))
display(pd.DataFrame(workbook_summary.items(), columns=["field", "value"]))

try:
    compiled = compile_workbook_file(optional_workbook_path)
except ValueError as exc:
    print("Expected compile behavior for default template policy:")
    print(exc)
else:
    print(
        "Workbook compiled successfully. "
        f"Trial specs count: {len(compiled.trial_specs)}"
    )

print("\nWorkbook-driven run is only valid after enabling/populating executable rows.")
print("Suggested command after workbook edits:")
print(
    " ".join(
        [
            *DECISION_SUPPORT_CMD,
            "--workbook",
            str(optional_workbook_path),
            "--experiment-id",
            SELECTED_EXPERIMENT_ID,
            "--index-csv",
            str(DATASET_INDEX_CSV),
            "--data-root",
            str(DATA_ROOT),
            "--cache-dir",
            str(NOTEBOOK_CACHE_DIR),
            "--output-root",
            str(NOTEBOOK_OUTPUT_ROOT),
            "--write-back-workbook",
        ]
    )
)

$ d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-workbook.EXE --output d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\tmp\workbook_appendix\generated_template.xlsx
[returncode] 0

[stdout]
Created workbook: D:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\tmp\workbook_appendix\generated_template.xlsx
Sheet order valid: True
Missing required sheets: None
Legacy required sheets present: True
New sheets present: True
Sheet count: 22
Data validations found: 56
Experiment_Definitions columns valid: True
Run_Log new columns present: True
Required named lists present: True
Missing named lists: None
Experiment_Ready formula present: True
Confirmatory formulas present: True
Dashboard formulas present: True
Stage vocabulary consistent: True
Stage rows detected: 7



### Generated workbook validation summary

,field,value
0,sheet_order_ok,True
1,missing_sheets,None
2,legacy_sheets_present,True
3,new_sheets_present,True
4,sheet_count,22
5,data_validations_found,56
6,experiment_definitions_columns_ok,True
7,run_log_new_columns_present,True
8,required_named_lists_present,True
9,missing_named_lists,None


Expected compile behavior for default template policy:
No enabled executable rows were found in Experiment_Definitions.

Workbook-driven run is only valid after enabling/populating executable rows.
Suggested command after workbook edits:
d:\Khaled\Projects\VScode\Thesis\.venv\Scripts\thesisml-run-decision-support.EXE --workbook d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\tmp\workbook_appendix\generated_template.xlsx --experiment-id E02 --index-csv d:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root d:\Khaled\Projects\VScode\Thesis\Data --cache-dir d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\cache --output-root d:\Khaled\Projects\VScode\Thesis\outputs\notebook_demo\end_to_end_experiment_walkthrough\campaign_outputs --write-back-workbook
